## NBA 3.1 | RX Patient Features

Builds First Occurrence, Recency, and Frequency features for RX claims.

**Input files (via `INPUT_PATHS_31` from `00_event_list_config`):**
- First Occurrence + Recency: `rx_all` (`rx_input_claims_total_events_and_first_occ_features_input`, 4.03B rows, 2021-01-02→2026-08-31)
- Frequency: `rx_12m` (`rx_input_claims_12M_features_input`, 2.55B rows, 2023-05-01→2026-08-31)

**Output:** `/Volumes/.../kerendia_nba_3_1_volume/notebook_output/patient_features/rx/`

**Skill:** generic-patient-features (Sections 3–5). Section 13 features are out of scope.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
import re
from pyspark.sql import functions as F, Window

# ── Feature family (claims only — do not change) ──────────────────────────────
FEATURE_FAMILY = "claims"
CLAIM_TYPE     = "rx"
EVENT_TYPE_VAL = "rx"

# ── Input paths (set by 00_event_list_config) ────────────────────────────────
RX_ALL_PATH  = INPUT_PATHS_31["rx_all"]   # First Occ + Recency
RX_12M_PATH  = INPUT_PATHS_31["rx_12m"]  # Frequency

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_BASE = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
               "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/rx")
FO_OUT    = f"{OUTPUT_BASE}/rx_first_occurrence_features"
REC_OUT   = f"{OUTPUT_BASE}/rx_recency_features"
FREQ_OUT  = f"{OUTPUT_BASE}/rx_frequency_features"

# ── Test-run sample (driven by job parameter; 0 = full universe) ─────────────
dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

# ── Lag / window parameters ──────────────────────────────────────────────────
RECENCY_WINDOWS   = [30, 60, 90, 180, 270, 360]  # days
LAG_WINDOWS       = [1, 2, 3, 6, 12]             # months (Freq Sub-B)
CHANGE_WINDOWS    = [3, 6]                        # months for log-ratio (Sub-C; NBA 3.1)
PATIENT_WINDOWS   = [30, 60, 90, 180, 360]        # days (Freq Sub-D)

print(f"OUTPUT_BASE : {OUTPUT_BASE}")
print(f"RX_ALL_PATH : {RX_ALL_PATH}")
print(f"RX_12M_PATH : {RX_12M_PATH}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()

# Section 10.8.6 checks
meta = cohorts.attrs["metadata"]
assert FEATURE_FAMILY == "claims" and meta["feature_family"] == "claims" and meta["offset_days"] == 0, \
    f"Wrong feature family or offset in cohort metadata: {meta}"
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)], "Expected 28 cohorts c1..c28"
c1 = cohorts.iloc[0]
assert (str(c1["lookback_start_date"]), str(c1["lookback_end_date"])) == ("2023-06-01", "2024-05-01"), \
    f"c1 dates mismatch: {c1['lookback_start_date']} / {c1['lookback_end_date']}"

print(f"Cohorts loaded: {len(cohorts)} rows")
print(cohorts[["cohort_id", "lookback_start_date", "lookback_end_date", "dataset"]].to_string(index=False))

In [0]:
def sanitize_column_name(name):
    name = re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = re.sub(r'_+', '_', name).strip('_')
    return name.upper()

def standardize(df, event_type):
    COLUMN_MAP = {
        "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
        "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
        "EVENT_NAME":  ["EVENT_NAME", "PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION",
                        "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
        "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
        "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
    }
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
                        "EVENT_TYPE", "CLAIM_ID", "DATA_SOURCE"] if c in df.columns]
    return df.select(*keep)

In [0]:
HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe")
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()

if TEST_HCP_SAMPLE > 0:
    hcp_universe = (hcp_universe
                    .orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID")
                    .limit(TEST_HCP_SAMPLE))

hcp_universe = F.broadcast(hcp_universe)
rx_event_keys = event_groups("rx")
print(f"Universe HCPs : {hcp_universe._jdf.count() if TEST_HCP_SAMPLE > 0 else '314,920 (full)'}")
print(f"Rx EVENT_GROUPs: {len(rx_event_keys)}")

In [0]:
claims_all = (spark.read.format("delta").load(RX_ALL_PATH)
              .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_all = standardize(claims_all, EVENT_TYPE_VAL)
validate_event_input(claims_all, CLAIM_TYPE)

# Early prune — keep only columns needed for First Occ + Recency
claims_all = claims_all.select(
    "PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE", "EVENT_TYPE", "CLAIM_ID")

In [0]:
# Step 1 — Pre-aggregate: earliest date per (HCP, event) across full history
pre = (claims_all
       .groupBy("PROVIDER_ID", "EVENT_NAME")
       .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE")))

# Step 2 — Pivot wide (cohort-independent)
fo_cols_raw = [sanitize_column_name(f"MAX_EVENT_TIME_{e}_FIRST_OCCURRENCE") for e in rx_event_keys]
wide = (pre.groupBy("PROVIDER_ID")
           .pivot("EVENT_NAME", rx_event_keys)
           .agg(F.min("FIRST_OCCURRENCE_DATE")))

# Rename pivot columns to canonical names
for orig, canon in zip(rx_event_keys, fo_cols_raw):
    safe_orig = sanitize_column_name(orig)
    if safe_orig in wide.columns:
        wide = wide.withColumnRenamed(safe_orig, canon)
    elif orig in wide.columns:
        wide = wide.withColumnRenamed(orig, canon)

# Step 3 — Cross-join with cohort anchor dates, compute EVENT_TIME, future-leakage guard
import pandas as pd
cohort_anchor_pd = cohorts[["cohort_id", "lookback_end_date"]].copy()
cohort_anchor_pd.columns = ["COHORT", "LOOKBACK_END_DATE"]
cohort_anchor_sp = F.broadcast(spark.createDataFrame(cohort_anchor_pd))

fo = wide.crossJoin(cohort_anchor_sp)

# Compute EVENT_TIME = datediff(last_day_of_end_month, col), null if future leak
# last_day_of_end_month = last day of LOOKBACK_END_DATE month
for canon, event in zip(fo_cols_raw, rx_event_keys):
    if canon in fo.columns:
        cutoff = F.last_day(F.col("LOOKBACK_END_DATE"))
        fo = fo.withColumn(
            canon,
            F.when(F.datediff(cutoff, F.col(canon)) >= 0,
                   F.datediff(cutoff, F.col(canon))).otherwise(F.lit(None))
        )

# Drop rows where every event column is NULL (no history for this HCP-cohort)
fo = fo.filter(F.greatest(*[F.col(c).isNotNull().cast("int") for c in fo_cols_raw if c in fo.columns]) == 1)
fo = fo.withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT"))

print(f"First Occurrence schema: {len([c for c in fo_cols_raw if c in fo.columns])} event cols")
fo.select("HCP_COHORT_ID", "COHORT").distinct().groupBy("COHORT").count().orderBy("COHORT").show(5, False)

In [0]:
(fo.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .partitionBy("COHORT")
   .save(FO_OUT))
print(f"Written: {FO_OUT}")

In [0]:
from functools import reduce
import math

rec_parts = []

for _, cohort_row in cohorts.iterrows():
    COHORT               = cohort_row["cohort_id"]
    LOOKBACK_START_DATE  = cohort_row["lookback_start_date"]
    LOOKBACK_END_DATE    = cohort_row["lookback_end_date"]
    last_day             = F.last_day(F.lit(LOOKBACK_END_DATE))

    # Filter to 12M window: >= start AND < add_months(end, 1)
    c_claims = (claims_all
                .filter((F.col("EVENT_DATE") >= F.lit(LOOKBACK_START_DATE)) &
                        (F.col("EVENT_DATE") < F.add_months(F.lit(LOOKBACK_END_DATE), 1)))
                .withColumn("EVENT_TIME", F.datediff(last_day, F.col("EVENT_DATE")))
                .withColumn("COHORT", F.lit(COHORT))
                .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.lit(COHORT))))

    # Sub-1: Most recent event per event name (min EVENT_TIME = fewest days before anchor)
    sub1 = (c_claims.groupBy("HCP_COHORT_ID")
                    .pivot("EVENT_NAME", rx_event_keys)
                    .agg(F.min("EVENT_TIME")))
    sub1_cols = {e: f"MIN_EVENT_TIME_{sanitize_column_name(e)}_TIME_MIN" for e in rx_event_keys}
    for e in rx_event_keys:
        safe = sanitize_column_name(e)
        if safe in sub1.columns:   sub1 = sub1.withColumnRenamed(safe, sub1_cols[e])
        elif e in sub1.columns:    sub1 = sub1.withColumnRenamed(e,    sub1_cols[e])

    # Sub-2: total events and unique events per HCP-cohort
    sub2 = (c_claims.groupBy("HCP_COHORT_ID")
                    .agg(F.count("CLAIM_ID").alias("TOTAL_EVENTS_RX"),
                         F.countDistinct("EVENT_NAME").alias("UNIQUE_EVENTS_RX")))

    # Sub-3: events in time windows
    sub3_aggs = []
    for w in RECENCY_WINDOWS:
        in_w = (c_claims.filter(F.col("EVENT_TIME") <= w)
                        .groupBy("HCP_COHORT_ID")
                        .agg(F.count("CLAIM_ID").alias(f"TOTAL_EVENTS_TIME_{w}_DAYS_RX"),
                             F.countDistinct("EVENT_NAME").alias(f"UNIQUE_EVENTS_TIME_{w}_DAYS_RX")))
        sub3_aggs.append(in_w)
    sub3 = reduce(lambda a, b: a.join(b, "HCP_COHORT_ID", "outer"), sub3_aggs)

    # Sub-6: skewness(EVENT_TIME) per type x window  
    sub6_aggs = []
    for w in RECENCY_WINDOWS:
        sk = (c_claims.filter(F.col("EVENT_TIME") <= w)
                      .groupBy("HCP_COHORT_ID")
                      .agg(F.skewness("EVENT_TIME").alias(f"SKEWNESS_TIME_{w}_DAYS_RX")))
        sub6_aggs.append(sk)
    sub6 = reduce(lambda a, b: a.join(b, "HCP_COHORT_ID", "outer"), sub6_aggs)

    # Join all sub-features for this cohort
    all_ids = c_claims.select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT").distinct()
    cohort_rec = (all_ids.join(sub1, "HCP_COHORT_ID", "left")
                         .join(sub2, "HCP_COHORT_ID", "left")
                         .join(sub3, "HCP_COHORT_ID", "left")
                         .join(sub6, "HCP_COHORT_ID", "left"))
    rec_parts.append(cohort_rec)

# Union all cohorts
recency = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), rec_parts)
print(f"Recency: {recency.count()} rows")
recency.groupBy("COHORT").count().orderBy("COHORT").show(5, False)

In [0]:
(recency.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("COHORT")
        .save(REC_OUT))
print(f"Written: {REC_OUT}")

In [0]:
claims_12m = (spark.read.format("delta").load(RX_12M_PATH)
               .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_12m = standardize(claims_12m, EVENT_TYPE_VAL)
claims_12m = (claims_12m
              .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
              .select("PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "SVC_DT_M", "CLAIM_ID"))
print(f"claims_12m rows (sample): {claims_12m.limit(5).count()} (lazy)")

In [0]:
# groupBy(PROVIDER_ID, SVC_DT_M).pivot(EVENT_NAME, event_keys).agg(count(lit(1)))
# Column naming: FREQ_{EVENT_NAME} — sanitized
freq_a_cols = {e: f"FREQ_{sanitize_column_name(e)}" for e in rx_event_keys}

freq_a = (claims_12m
          .groupBy("PROVIDER_ID", "SVC_DT_M")
          .pivot("EVENT_NAME", rx_event_keys)
          .agg(F.count(F.lit(1))))

for e in rx_event_keys:
    safe = sanitize_column_name(e)
    if safe in freq_a.columns:   freq_a = freq_a.withColumnRenamed(safe, freq_a_cols[e])
    elif e in freq_a.columns:    freq_a = freq_a.withColumnRenamed(e,    freq_a_cols[e])

freq_a = freq_a.fillna(0, subset=[v for v in freq_a_cols.values() if v in freq_a.columns])
print(f"Freq Sub-A schema: {len(freq_a.columns)} columns")

In [0]:
from pyspark.sql.functions import col

freq_cols = [c for c in freq_a.columns if c not in ("PROVIDER_ID", "SVC_DT_M")]

# Dense spine: all HCPs x all months in range
min_m = claims_12m.agg(F.min("SVC_DT_M")).first()[0]
max_m = claims_12m.agg(F.max("SVC_DT_M")).first()[0]
months_range = spark.range(0, 1000).select(
    F.expr(f"date_add(date '{min_m}', cast(id * 30 as int))").alias("_d")) \
    .filter(F.col("_d") <= F.lit(max_m)) \
    .select(F.trunc("_d", "month").alias("SVC_DT_M")).distinct()

all_hcps = freq_a.select("PROVIDER_ID").distinct()
spine = all_hcps.crossJoin(months_range)

freq_b = (spine.join(freq_a, ["PROVIDER_ID", "SVC_DT_M"], "left")
               .fillna(0, subset=freq_cols))

# Rolling windows via Window.rowsBetween
w_spec = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
lag_aggs = {}
for col_name in freq_cols:
    for n in LAG_WINDOWS:
        alias = f"{col_name}_IN_LAST_{n}_MONTH"
        lag_aggs[alias] = F.sum(col_name).over(w_spec.rowsBetween(-(n - 1), 0))

# Single projection: keys + all lag columns (raw FREQ_* columns dropped implicitly)
freq_b = freq_b.select("PROVIDER_ID", "SVC_DT_M",
                       *[expr.alias(alias) for alias, expr in lag_aggs.items()])
print(f"Freq Sub-B: {len([c for c in freq_b.columns if '_IN_LAST_' in c])} lag columns")

In [0]:
import math

existing_cols = set(freq_b.columns)
logratio_exprs = []
for col_name in [f"FREQ_{sanitize_column_name(e)}" for e in rx_event_keys]:
    f12 = f"{col_name}_IN_LAST_12_MONTH"
    if f12 not in existing_cols:
        continue
    for n in CHANGE_WINDOWS:
        fn  = f"{col_name}_IN_LAST_{n}_MONTH"
        if fn not in existing_cols:
            continue
        alias = f"LOGRATIO_FREQ_{col_name.replace('FREQ_', '')}_IN_LAST_{n}_MONTH"
        logratio_exprs.append(
            F.log(((F.col(fn) / n) + F.lit(0.5)) /
                  (((F.col(f12) - F.col(fn)) / (12 - n)) + F.lit(0.5))).alias(alias))

freq_b = freq_b.select("*", *logratio_exprs)

print(f"Freq Sub-C added: {len([c for c in freq_b.columns if c.startswith('LOGRATIO')])} columns")

In [0]:
pat_parts = []

for _, cohort_row in cohorts.iterrows():
    COHORT              = cohort_row["cohort_id"]
    LOOKBACK_START_DATE = cohort_row["lookback_start_date"]
    LOOKBACK_END_DATE   = cohort_row["lookback_end_date"]
    last_day_expr       = F.last_day(F.lit(LOOKBACK_END_DATE))

    c_pat = (claims_12m
             .filter((F.col("SVC_DT_M") >= F.lit(LOOKBACK_START_DATE)) &
                     (F.col("SVC_DT_M") < F.add_months(F.lit(LOOKBACK_END_DATE), 1)))
             .withColumn("DAYS_DIFF",
                         F.datediff(last_day_expr,
                                    F.last_day(F.col("SVC_DT_M")))))

    sub_d_aggs = []
    for w in PATIENT_WINDOWS:
        flag = f"_IN_{w}D"
        in_w = (c_pat.filter(F.col("DAYS_DIFF") <= w)
                     .dropDuplicates(["PROVIDER_ID", "EVENT_NAME", "PATIENT_ID"])
                     .groupBy("PROVIDER_ID", "EVENT_NAME")
                     .agg(F.countDistinct("PATIENT_ID").alias("_cnt")))
        pivoted = (in_w.groupBy("PROVIDER_ID")
                       .pivot("EVENT_NAME", rx_event_keys)
                       .agg(F.max("_cnt")))
        for e in rx_event_keys:
            safe = sanitize_column_name(e)
            new_col = f"NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{safe}_IN_LAST_{w}_DAYS"
            if safe in pivoted.columns:  pivoted = pivoted.withColumnRenamed(safe, new_col)
            elif e in pivoted.columns:   pivoted = pivoted.withColumnRenamed(e, new_col)
        sub_d_aggs.append(pivoted)

    sub_d = reduce(lambda a, b: a.join(b, "PROVIDER_ID", "outer"), sub_d_aggs)
    sub_d = sub_d.withColumn("COHORT", F.lit(COHORT))
    pat_parts.append(sub_d)

freq_d = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), pat_parts)
print(f"Freq Sub-D: {freq_d.count()} rows")

In [0]:
# Join Sub-B/C (monthly grain) to cohort anchor dates → add COHORT column
cohort_svc_map_pd = []
for _, row in cohorts.iterrows():
    cohort_svc_map_pd.append({
        "COHORT": row["cohort_id"],
        "SVC_DT_M": row["lookback_end_date"]  # only the last month links to the cohort
    })
# Keep all months: join on SVC_DT_M in range [lookback_start, lookback_end)
# For the purpose of writing, we attach cohort info using the last SVC_DT_M per cohort.
# Full monthly grain is retained in the output; COHORT is the partition key.
cohort_range_pd = []
for _, row in cohorts.iterrows():
    start = row["lookback_start_date"]
    end   = row["lookback_end_date"]
    cohort_range_pd.append({
        "COHORT": row["cohort_id"],
        "COHORT_MONTH_START": start,
        "COHORT_MONTH_END":   end,
    })
cohort_range_sp = spark.createDataFrame(cohort_range_pd)

# Each SVC_DT_M belongs to a cohort when it's in [lookback_start, lookback_end)
freq_bc_cohort = (freq_b
    .crossJoin(F.broadcast(cohort_range_sp))
    .filter((F.col("SVC_DT_M") >= F.col("COHORT_MONTH_START")) &
            (F.col("SVC_DT_M") < F.add_months(F.col("COHORT_MONTH_END"), 1)))
    .drop("COHORT_MONTH_START", "COHORT_MONTH_END"))

# Join Sub-D (HCP x COHORT grain)
frequency = (freq_bc_cohort
             .join(freq_d, ["PROVIDER_ID", "COHORT"], "left"))

(frequency.write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .partitionBy("COHORT")
          .save(FREQ_OUT))
print(f"Written: {FREQ_OUT}")
print(f"Row count: {frequency.count()}")

In [0]:
for path, label in [(FO_OUT, "First Occurrence"), (REC_OUT, "Recency"), (FREQ_OUT, "Frequency")]:
    df = spark.read.format("delta").load(path)
    cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
    expected   = set(cohorts["cohort_id"])
    extra      = cohort_ids - expected
    missing    = expected - cohort_ids
    n_outside  = (df.select("PROVIDER_ID").distinct()
                    .join(hcp_universe, "PROVIDER_ID", "left_anti").count())
    print(f"{label}: {df.count():,} rows | cohorts={len(cohort_ids)}/28 "
          f"| extra={extra or 'none'} | missing={missing or 'none'} "
          f"| HCPs outside universe={n_outside}")
dbutils.notebook.exit("RX patient features written: First Occurrence, Recency, Frequency")